In [2]:
from qiskit_qaoa.utils.hamiltonian_utils import get_Q_and_hamiltonian
from qiskit_ibm_runtime import QiskitRuntimeService

In [3]:
filename = 'test_N7_W4'

In [4]:
data_file = f'/lustre/scratch127/qpg/jc59/new_qubo_formulation/oriented/qubo_data/qubo_data_{filename}.gfa.pkl'

Q, hamiltonian, offset, ising_offset = get_Q_and_hamiltonian(data_file)
num_qubits: int = hamiltonian.num_qubits

# service = QiskitRuntimeService(name='eu_test_instance')
# backend = service.least_busy(min_num_qubits=num_qubits, operational=True, simulator=False) 
service = QiskitRuntimeService(name='us_instance')
backend_grid = service.backend(name='ibm_miami')
backend_hex = service.backend(name='ibm_boston')

In [5]:
from qubo_qaoa.utils.swap_strategy import QUBOSwapStrategy
import networkx as nx
import re
from itertools import combinations


In [53]:
rows, cols = 1, 1
while 4 * (rows + cols + rows * cols) < num_qubits:
    if rows < cols:
        rows += 1
    else:
        cols += 1
print(f'Min size to support virtual qubits: {(rows, cols)}')

swap_strat = QUBOSwapStrategy.from_heavy_hex(rows, cols)
coupling_map = swap_strat._coupling_map
coupling_map_edge = list(coupling_map)
physical_qubits = list(coupling_map.physical_qubits)

dual_coupling_map = nx.Graph()

for qubit in physical_qubits:
    edges = [edge for edge in coupling_map_edge if edge[0]==qubit]
    for edge1, edge2 in combinations(edges, 2):
        dual_coupling_map.add_edge(tuple(sorted(edge1)), tuple(sorted(edge2)))
edge_colouring = nx.greedy_color(dual_coupling_map, interchange=True)
edge_colouring_copy = {}
for k, v in edge_colouring.items():
    edge_colouring_copy[k] = v
    edge_colouring_copy[k[::-1]] = v
edge_colouring = edge_colouring_copy

Min size to support virtual qubits: (3, 3)


In [7]:
from qiskit_qaoa.utils.circuit_graph_utils import circuit_to_graph, graph_to_operator
from qiskit.circuit.library import QAOAAnsatz
from qopt_best_practices.sat_mapping import SATMapper
import numpy as np


In [8]:
qc = QAOAAnsatz(
    cost_operator=hamiltonian,
    reps = 1,
    flatten=True
)
graph = circuit_to_graph(qc, qc.parameters[1])

remapped_g, sat_map, min_sat_layers = SATMapper(timeout=60).remap_graph_with_sat(
    graph=graph, swap_strategy=swap_strat, max_layers = int(num_qubits + np.sqrt(num_qubits) + 61)
)
if remapped_g is None or sat_map is None:
    raise Exception('Failed to find initial layout')

cost_op_hex = graph_to_operator(remapped_g, swap_strat._num_vertices)

CNF size: 276816
CNF size: 276816
CNF size: 276816
CNF size: 276816
CNF size: 276816
CNF size: 276816
CNF size: 276816


In [9]:
from qubo_qaoa.utils.lr_qaoa import get_hardware_LR_qaoa_circuit
from qiskit.circuit import ParameterVector


In [10]:
phis = ParameterVector('ϕ', num_qubits)
delta_b = 0.63
delta_g = 0.16
p = 1


In [11]:
fixed_qc, circuit_hex = get_hardware_LR_qaoa_circuit(
    p, delta_b, delta_g, num_qubits,
    cost_op_hex, sat_map, backend_hex, edge_colouring, swap_strat,
    None, phis=phis,
)

Constructing circuit
Transpiling doubles
Transpiling circuit


In [12]:
circuit_hex.count_ops()

OrderedDict([('sx', 8812),
             ('rz', 7070),
             ('cz', 4603),
             ('delay', 1783),
             ('measure', 56),
             ('x', 36)])

In [55]:
print('Constructing circuit')
num_virtual_qubits = 56
n = swap_strat._num_vertices

singles = cost_op_hex[cost_op_hex.paulis.z.sum(axis=-1) == 1]
doubles = cost_op_hex[cost_op_hex.paulis.z.sum(axis=-1) == 2]
doubles_circ = QAOAAnsatz(
    doubles,
    initial_state=QuantumCircuit(n),
    mixer_operator=QuantumCircuit(n)
)
config = {
    "num_layers": 1,
    "swap_strategy": swap_strat,
    "edge_coloring": edge_colouring,
    "construct_qaoa": False,
    "basis_gates": ["rz", "rzz", "cz", "cx", "x", "swap"]
}

properties = {}
def get_permutation(pass_, dag, time, property_set, count):
    properties["virtual_permutation_layout"] = property_set["virtual_permutation_layout"]
pm = qaoa_swap_strategy_pm(config)
print('Transpiling doubles')
tdoubles_circ: QuantumCircuit = pm.run(doubles_circ, callback=get_permutation)

singles_circ = QuantumCircuit(n)
singles_circ.append(PauliEvolutionGate(singles, time=tdoubles_circ.parameters[0]), range(n))
tsingles: QuantumCircuit = transpile(singles_circ, basis_gates=["rz"])
cost_circ: QuantumCircuit = tsingles.compose(tdoubles_circ, inplace=False)

init_state = QuantumCircuit(n)
if phis is not None:
    if not len(phis) == 56:
        raise Exception(f'Wrong number of phis, expected {56}, got {len(phis)}')
    for i in range(num_virtual_qubits):
        qubit = sat_map[i]
        init_state.ry(phis[i], qubit)
else:
    init_state.h([x for x in sat_map.values()])  
    
    
mixer_layer_even = QuantumCircuit(n)
beta = Parameter("β")
if phis is not None:
    inv_sat_map = {v: k for k, v in sat_map.items()}
    for i, qidx in [(inv_sat_map[x], properties['virtual_permutation_layout'].get_physical_bits()[x]) for x in sat_map.values()]:
        mixer_layer_even.ry(-phis[i], qidx)
        mixer_layer_even.rz(-2 * beta, qidx)
        mixer_layer_even.ry(phis[i], qidx)
else:
    mixer_layer_even.rx(-2 * beta, [properties['virtual_permutation_layout'].get_physical_bits()[x] for x in sat_map.values()])


mixer_layer_odd = QuantumCircuit(n)
if phis is not None:
    for i in range(num_virtual_qubits):
        qubit = sat_map[i]
        mixer_layer_odd.ry(-phis[i], qubit)
        mixer_layer_odd.rz(-2 * beta, qubit)
        mixer_layer_odd.ry(phis[i], qubit)
else:
    mixer_layer_odd.rx(-2 * beta, [x for x in sat_map.values()])

gammas = ParameterVector("γ",p)
betas = ParameterVector("β", p)

qaoa_circuit = QuantumCircuit(n, num_virtual_qubits)

qaoa_circuit.compose(init_state, inplace=True)
for layer in range(p):
    bind_dict = {cost_circ.parameters[0]: gammas[layer]}
    bound_cost_layer = cost_circ.assign_parameters(bind_dict)

    mixer_layer = mixer_layer_even if layer % 2 == 0 else mixer_layer_odd
    bind_dict = {mixer_layer.parameters[0]: betas[layer]}
    bound_mixer_layer = mixer_layer.assign_parameters(bind_dict)

    if layer % 2 == 0:
        # even layer -> append cost
        qaoa_circuit.compose(bound_cost_layer, range(n), inplace=True)
    else:
        # odd layer -> append reversed cost
        qaoa_circuit.compose(
            bound_cost_layer.reverse_ops(), range(n), inplace=True
        )
    qaoa_circuit.compose(bound_mixer_layer, range(n), inplace=True)

if p % 2 == 1:
    # iterate over layout permutations to recover measurements
    if properties["virtual_permutation_layout"]:
        inv_sat_map = {v: k for k, v in sat_map.items()}
        for cidx, qidx in [(inv_sat_map[x], properties['virtual_permutation_layout'].get_physical_bits()[x]) for x in sat_map.values()]:
            qaoa_circuit.measure(qidx, cidx)
    else:
        print("layout not found, assigining trivial layout")
        for cidx, qidx in sat_map.items():
            qaoa_circuit.measure(qidx, cidx)
else:
    for cidx, qidx in sat_map.items():
        qaoa_circuit.measure(qidx, cidx)

Constructing circuit
Transpiling doubles


In [56]:
qaoa_circuit.count_ops()

OrderedDict([('swap', 1409),
             ('rzz', 598),
             ('ry', 168),
             ('rz', 112),
             ('measure', 56)])

In [13]:
matches = re.findall(r'[0-9]+', filename)  
if len(matches) == 2:
    rows, cols = 2*int(matches[0]), int(matches[1])
else:
    raise Exception('No rows and cols')
swap_strat_grid = QUBOSwapStrategy.from_grid(rows, cols)
qubits = [(row, col) for row in range(rows) for col in range(cols)]
mapping = {qubit: idx for idx, qubit in enumerate(qubits)}
edge_colouring = {}
for r in range(rows):
    for c in range(cols):
        if c + 1 < cols:
            a = (r, c)
            b = (r, c + 1)
            layer = 0 if (c % 2 == 0) else 1  # horizontal: left column parity
            edge_colouring[(mapping[a], mapping[b])] = layer
            edge_colouring[(mapping[b], mapping[a])] = layer
        if r + 1 < rows:
            a = (r, c)
            b = (r + 1, c)
            layer = 2 if (r % 2 == 0) else 3  # vertical: top row parity
            edge_colouring[(mapping[a], mapping[b])] = layer
            edge_colouring[(mapping[b], mapping[a])] = layer

In [14]:
remapped_g_grid, sat_map_grid, min_sat_layers_grid = SATMapper(timeout=60).remap_graph_with_sat(
    graph=graph, swap_strategy=swap_strat_grid, max_layers = int(num_qubits + np.sqrt(num_qubits) + 61)
)
if remapped_g is None or sat_map is None:
    raise Exception('Failed to find initial layout')

cost_op_grid = graph_to_operator(remapped_g_grid, swap_strat_grid._num_vertices)

CNF size: 209160
CNF size: 209160
CNF size: 209160
CNF size: 209160
CNF size: 209160
CNF size: 209160
CNF size: 209160


In [18]:
swap_strat_grid._num_vertices

56

In [19]:
fixed_qc, circuit_grid = get_hardware_LR_qaoa_circuit(
    p, delta_b, delta_g, num_qubits,
    cost_op_grid, sat_map_grid, backend_grid, edge_colouring, swap_strat_grid,
    None, phis=phis,
)

Constructing circuit
Transpiling doubles
Transpiling circuit


In [20]:
circuit_grid.count_ops()

OrderedDict([('sx', 7169),
             ('rz', 5810),
             ('cz', 3906),
             ('delay', 2074),
             ('measure', 56),
             ('x', 3)])

In [31]:
from qiskit_aer import AerSimulator
from qiskit_aer.backends.backendconfiguration import AerBackendConfiguration
backend_options = dict(
    method="statevector",
    device='CPU',
    precision='single',
    basis_gates=["sx", "x", "rz", "rzz", "cz", "id", "cx", "swap"]
)

In [36]:
config = AerSimulator._DEFAULT_CONFIGURATION
config["n_qubits"] = 56
config = AerBackendConfiguration.from_dict(config)
cm = swap_strat_grid._coupling_map
cm.make_symmetric()
backend_grid_swap = AerSimulator(configuration=config, coupling_map=cm, **backend_options)
backend_grid_swap.set_option("n_qubits", 56)

In [42]:
len(list(cm.connected_components()[0]))

188

In [44]:

from qiskit import QuantumCircuit, transpile
from qiskit.circuit import Parameter, ParameterVector
from qiskit.circuit.library import QAOAAnsatz, PauliEvolutionGate
from qopt_best_practices.transpilation import qaoa_swap_strategy_pm


In [51]:
print('Constructing circuit')
num_virtual_qubits = 56
n = swap_strat_grid._num_vertices

singles = cost_op_grid[cost_op_grid.paulis.z.sum(axis=-1) == 1]
doubles = cost_op_grid[cost_op_grid.paulis.z.sum(axis=-1) == 2]
doubles_circ = QAOAAnsatz(
    doubles,
    initial_state=QuantumCircuit(n),
    mixer_operator=QuantumCircuit(n)
)
config = {
    "num_layers": 1,
    "swap_strategy": swap_strat_grid,
    "edge_coloring": edge_colouring,
    "construct_qaoa": False,
    "basis_gates": ["rz", "rzz", "cz", "cx", "x", "swap"]
}

properties = {}
def get_permutation(pass_, dag, time, property_set, count):
    properties["virtual_permutation_layout"] = property_set["virtual_permutation_layout"]
pm = qaoa_swap_strategy_pm(config)
print('Transpiling doubles')
tdoubles_circ: QuantumCircuit = pm.run(doubles_circ, callback=get_permutation)

singles_circ = QuantumCircuit(n)
singles_circ.append(PauliEvolutionGate(singles, time=tdoubles_circ.parameters[0]), range(n))
tsingles: QuantumCircuit = transpile(singles_circ, basis_gates=["rz"])
cost_circ: QuantumCircuit = tsingles.compose(tdoubles_circ, inplace=False)

init_state = QuantumCircuit(n)
if phis is not None:
    if not len(phis) == 56:
        raise Exception(f'Wrong number of phis, expected {56}, got {len(phis)}')
    for i in range(num_virtual_qubits):
        qubit = sat_map_grid[i]
        init_state.ry(phis[i], qubit)
else:
    init_state.h([x for x in sat_map_grid.values()])  
    
    
mixer_layer_even = QuantumCircuit(n)
beta = Parameter("β")
if phis is not None:
    inv_sat_map = {v: k for k, v in sat_map_grid.items()}
    for i, qidx in [(inv_sat_map[x], properties['virtual_permutation_layout'].get_physical_bits()[x]) for x in sat_map_grid.values()]:
        mixer_layer_even.ry(-phis[i], qidx)
        mixer_layer_even.rz(-2 * beta, qidx)
        mixer_layer_even.ry(phis[i], qidx)
else:
    mixer_layer_even.rx(-2 * beta, [properties['virtual_permutation_layout'].get_physical_bits()[x] for x in sat_map_grid.values()])


mixer_layer_odd = QuantumCircuit(n)
if phis is not None:
    for i in range(num_virtual_qubits):
        qubit = sat_map_grid[i]
        mixer_layer_odd.ry(-phis[i], qubit)
        mixer_layer_odd.rz(-2 * beta, qubit)
        mixer_layer_odd.ry(phis[i], qubit)
else:
    mixer_layer_odd.rx(-2 * beta, [x for x in sat_map_grid.values()])

gammas = ParameterVector("γ",p)
betas = ParameterVector("β", p)

qaoa_circuit = QuantumCircuit(n, num_virtual_qubits)

qaoa_circuit.compose(init_state, inplace=True)
for layer in range(p):
    bind_dict = {cost_circ.parameters[0]: gammas[layer]}
    bound_cost_layer = cost_circ.assign_parameters(bind_dict)

    mixer_layer = mixer_layer_even if layer % 2 == 0 else mixer_layer_odd
    bind_dict = {mixer_layer.parameters[0]: betas[layer]}
    bound_mixer_layer = mixer_layer.assign_parameters(bind_dict)

    if layer % 2 == 0:
        # even layer -> append cost
        qaoa_circuit.compose(bound_cost_layer, range(n), inplace=True)
    else:
        # odd layer -> append reversed cost
        qaoa_circuit.compose(
            bound_cost_layer.reverse_ops(), range(n), inplace=True
        )
    qaoa_circuit.compose(bound_mixer_layer, range(n), inplace=True)

if p % 2 == 1:
    # iterate over layout permutations to recover measurements
    if properties["virtual_permutation_layout"]:
        inv_sat_map = {v: k for k, v in sat_map_grid.items()}
        for cidx, qidx in [(inv_sat_map[x], properties['virtual_permutation_layout'].get_physical_bits()[x]) for x in sat_map_grid.values()]:
            qaoa_circuit.measure(qidx, cidx)
    else:
        print("layout not found, assigining trivial layout")
        for cidx, qidx in sat_map_grid.items():
            qaoa_circuit.measure(qidx, cidx)
else:
    for cidx, qidx in sat_map_grid.items():
        qaoa_circuit.measure(qidx, cidx)

Constructing circuit
Transpiling doubles


In [52]:
qaoa_circuit.count_ops()

OrderedDict([('swap', 617),
             ('rzz', 598),
             ('ry', 168),
             ('rz', 112),
             ('measure', 56)])